# 09 - Patch-Based U-Net

Notebook 04's Random Forest is a real baseline, but pixel-wise classification is not the scalable path: it took 14 minutes to train on pixels from just 15 of 446 chips, and inference means classifying every pixel independently with no shared computation. `utils/ai/classic/unet.py` (written earlier, never trained until now) processes a whole chip in one batched forward pass through convolutions - the actual credible answer to "efficient and scalable computational strategies" for full-scene segmentation.

Uses `utils/ai/classic/sen1floods11_dataset.Sen1Floods11Dataset` - a real PyTorch `Dataset` over the same chips, `MaskedComboLoss` (`utils/ai/classic/losses.py`, added for this run) to correctly exclude sen1floods11's `-1` no-data pixels from the loss.

In [1]:
import os
import sys

# Portable project-root resolution (no machine-specific hardcoded path) -
# walk up from the current working directory until pyproject.toml is found.
# One statement on purpose: ruff/pycodestyle's E402 ("imports not at top")
# specifically exempts a lone sys.path.insert(...) call, not a multi-
# statement block before it.
sys.path.insert(0, next(
    d for d in (
        os.path.abspath(os.path.join(os.getcwd(), *([os.pardir] * i)))
        for i in range(8)
    )
    if os.path.exists(os.path.join(d, "pyproject.toml"))
))

import time

import torch
from torch.utils.data import DataLoader

from utils.ai.classic.losses import MaskedComboLoss
from utils.ai.classic.sen1floods11_dataset import Sen1Floods11Dataset
from utils.ai.classic.unet import UNet
from utils.ai.objectives.registry import evaluate
from utils.observability.run_logger import RunLogger

_root = sys.path[0]
logger = RunLogger("09_patch_unet")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")

device: cpu


## Data

Full 512x512 chips, 2 channels (VV, VH) - no pixel flattening, no subsampling to a handful of points. Kept to a modest chip count and epoch count to finish in a reasonable time on CPU, same honesty-about-scope as the RF/QML notebooks - the point is to demonstrate the batched-tensor-op scaling advantage is real, not to produce a publication-grade model.

In [2]:
N_TRAIN_CHIPS = 20
N_VAL_CHIPS = 8
BATCH_SIZE = 2
EPOCHS = 3

with logger.stage("build_datasets") as stage:
    train_ds = Sen1Floods11Dataset("train")
    val_ds = Sen1Floods11Dataset("valid")
    train_subset = torch.utils.data.Subset(train_ds, range(min(N_TRAIN_CHIPS, len(train_ds))))
    val_subset = torch.utils.data.Subset(val_ds, range(min(N_VAL_CHIPS, len(val_ds))))
    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE)
    stage.metrics = {"n_train_chips": len(train_subset), "n_val_chips": len(val_subset)}

sample = train_subset[0]
print(f"one sample: x={sample['x'].shape} y={sample['y'].shape} valid_mask={sample['valid_mask'].shape}")

[09_patch_unet] -> build_datasets ...
[09_patch_unet] <- build_datasets [OK] 0.004s {'n_train_chips': 20, 'n_val_chips': 8}


one sample: x=torch.Size([2, 512, 512]) y=torch.Size([512, 512]) valid_mask=torch.Size([512, 512])


In [3]:
model = UNet(in_channels=2, num_classes=1, base_channels=16).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = MaskedComboLoss()

n_params = sum(p.numel() for p in model.parameters())
print(f"model: {n_params:,} parameters")

model: 1,943,905 parameters


In [4]:
with logger.stage("train_unet") as stage:
    epoch_times = []
    for epoch in range(EPOCHS):
        epoch_start = time.time()
        model.train()
        train_loss = 0.0
        n_chips_seen = 0
        for batch in train_loader:
            x = batch["x"].to(DEVICE)
            y = batch["y"].unsqueeze(1).to(DEVICE)
            mask = batch["valid_mask"].unsqueeze(1).to(DEVICE)

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y, mask)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * x.size(0)
            n_chips_seen += x.size(0)

        epoch_time = time.time() - epoch_start
        epoch_times.append(epoch_time)
        throughput = n_chips_seen / epoch_time
        print(f"epoch {epoch+1}/{EPOCHS} loss={train_loss/n_chips_seen:.4f} "
              f"time={epoch_time:.1f}s throughput={throughput:.2f} chips/s (512x512 each, full conv forward+backward)")

    stage.metrics = {
        "epochs": EPOCHS,
        "mean_epoch_s": round(sum(epoch_times) / len(epoch_times), 1),
        "chips_per_sec": round(N_TRAIN_CHIPS / (sum(epoch_times) / len(epoch_times)), 3),
    }

[09_patch_unet] -> train_unet ...


epoch 1/3 loss=0.7602 time=104.9s throughput=0.19 chips/s (512x512 each, full conv forward+backward)


epoch 2/3 loss=0.7299 time=120.4s throughput=0.17 chips/s (512x512 each, full conv forward+backward)


epoch 3/3 loss=0.6937 time=123.6s throughput=0.16 chips/s (512x512 each, full conv forward+backward)
[09_patch_unet] <- train_unet [OK] 348.988s {'epochs': 3, 'mean_epoch_s': 116.3, 'chips_per_sec': 0.172}


## Evaluate

Real held-out chips, real metrics, plus inference throughput - the actual scalability claim (chips/sec on a single CPU core, batchable and parallelizable further on GPU/multi-core, unlike the pixel-wise RF).

In [5]:
with logger.stage("evaluate_unet") as stage:
    model.eval()
    all_metrics = []
    infer_start = time.time()
    n_infer_chips = 0
    with torch.no_grad():
        for batch in val_loader:
            x = batch["x"].to(DEVICE)
            y = batch["y"].numpy()
            mask = batch["valid_mask"].numpy()

            logits = model(x)
            preds = (torch.sigmoid(logits).squeeze(1).cpu().numpy() > 0.5).astype(int)
            n_infer_chips += x.size(0)

            for i in range(preds.shape[0]):
                valid = mask[i]
                m = evaluate("flood-segmentation", preds[i][valid], y[i][valid].astype(int))
                all_metrics.append(m)

    infer_time = time.time() - infer_start
    infer_throughput = n_infer_chips / infer_time

    mean_metrics = {k: sum(m[k] for m in all_metrics) / len(all_metrics) for k in all_metrics[0]}
    stage.metrics = {**{k: round(v, 4) for k, v in mean_metrics.items()}, "infer_chips_per_sec": round(infer_throughput, 3)}

print(f"mean validation metrics over {len(all_metrics)} chips:")
for k, v in mean_metrics.items():
    print(f"  {k:12s} {v:.4f}")
print(f"inference throughput: {infer_throughput:.2f} chips/s ({512*512} px/chip) = {infer_throughput*512*512:,.0f} px/s")
print(f"compare: notebook 04's pixel-wise RF trained on 3.2M pixels took 848s -> {3229891/848:.0f} px/s for TRAINING alone,")
print("not a fair apples-to-apples (train vs inference), but illustrates the batched-tensor-op vs per-pixel-classifier gap.")

[09_patch_unet] -> evaluate_unet ...


[09_patch_unet] <- evaluate_unet [OK] 9.091s {'iou': 0.625, 'f1': 0.625, 'precision': 1.0, 'recall': 0.625, 'boundary_f1': 0.625, 'infer_chips_per_sec': 0.88}
mean validation metrics over 8 chips:
  iou          0.6250
  f1           0.6250
  precision    1.0000
  recall       0.6250
  boundary_f1  0.6250
inference throughput: 0.88 chips/s (262144 px/chip) = 230,772 px/s
compare: notebook 04's pixel-wise RF trained on 3.2M pixels took 848s -> 3809 px/s for TRAINING alone,
not a fair apples-to-apples (train vs inference), but illustrates the batched-tensor-op vs per-pixel-classifier gap.


In [6]:
import os

MODEL_PATH = os.path.join(_root, "datasets", "processed", "models", "patch_unet_v1.pt")
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
torch.save(model.state_dict(), MODEL_PATH)

logger.log_metrics({"model_path": MODEL_PATH, **{f"valid_{k}": v for k, v in mean_metrics.items()}})
logger.finalize()
print(f"saved model -> {MODEL_PATH}")

[09_patch_unet] run complete in 362.222s -> D:\project-raw-data\sphoorthq-geoverse\datasets\reports\runs\26640281-f33a-41b5-91ee-46ebb832ee8c.json
saved model -> d:\project-raw-data\sphoorthq-geoverse\datasets\processed\models\patch_unet_v1.pt
